# Biology for Computer Scientists

> **Portfolio artifact:** `portfolio/module05_biology_for_computer_scientists.ipynb`  
> **Target audience:** someone with strong Python and statistics, zero formal biology  
> **Pass standard:** publishable without embarrassment; no unexplained jargon; every analogy holds

*by Vinoth* — *Module 05 Mini-Project 02, 12-month Computational Biology Research Notebook*

---

This notebook explains biology using the mental models that computer scientists already have.
No lab experience required. No memorization of taxonomic hierarchies. By the end, you
should be able to read a genomics paper abstract and follow the biological argument.

---
## 1. The Cell as a Factory

All living things are made of **cells**. Every cell runs the same basic software on
similar hardware, with slight variations that make a liver cell different from a neuron.

The analogy is a **distributed computing node**:

| Cell component | Computing analogy | What it actually does |
|---------------|------------------|-----------------------|
| **Nucleus** | Persistent storage (HDD) | Houses the DNA — all the blueprints |
| **Ribosome** | CPU / assembly line | Executes instructions to build proteins |
| **Mitochondria** | Power supply (PSU) | Generates ATP (chemical energy currency) |
| **Plasma membrane** | Network interface + firewall | Controls what enters and leaves |
| **Endoplasmic reticulum** | Compile + link step | Folds and processes proteins before use |
| **Golgi apparatus** | Package manager / post office | Routes finished proteins to their destination |

**Why this matters for data:**
When an RNA-seq experiment produces a count matrix, each row is a gene being expressed
in a particular cell. You're essentially reading the memory log of the factory at a
snapshot in time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Cell diagram: factory analogy
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('The Cell as a Factory', fontsize=14, fontweight='bold')

# Factory walls
factory = patches.Ellipse((5, 4), 9.5, 7.5, fill=True, facecolor='#f5f5e8',
                            edgecolor='#8b8b3a', lw=3)
ax.add_patch(factory)

components = [
    ((5, 4.2), 2.5, 2.0, '#b3d9ff', '#336699', 'Nucleus\n(Persistent storage\nDNA blueprints)'),
    ((1.5, 6.2), 1.8, 1.0, '#ffd9b3', '#996633', 'Mitochondria\n(Power supply)'),
    ((8.5, 6.0), 1.8, 1.0, '#d4b8e0', '#663399', 'Ribosome\n(CPU: builds proteins)'),
    ((1.8, 2.5), 2.0, 1.0, '#b3ffb3', '#336633', 'ER\n(Compile + link)'),
    ((8.2, 2.5), 1.8, 1.0, '#ffe680', '#666600', 'Golgi\n(Post office)'),
]
for (x, y), w, h, fc, ec, label in components:
    rect = patches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                                   boxstyle='round,pad=0.1', facecolor=fc, edgecolor=ec, lw=1.5)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=7.5)

ax.text(5, 0.3, 'Plasma membrane (Network interface + firewall)', ha='center',
        fontsize=9, color='#8b8b3a', style='italic')
plt.tight_layout()
plt.show()

---
## 2. The Central Dogma as a Data Pipeline

The single most important concept in molecular biology:

```
DNA  ─────[transcription]────►  mRNA  ─────[translation]────►  Protein
 │                                │                               │
Source code                   Runtime                        Executable
(stored, heritable,          (temporary, cell-specific,     (does the actual
 never directly executed)      interpreted by ribosomes)      work in the cell)
```

**The pipeline in detail:**

1. **DNA** is double-stranded and stored in the nucleus. Think of it as version-controlled
   source code — never directly executed, never modified during normal operation.
2. **Transcription** copies a gene's DNA into a temporary mRNA molecule. The cell only
   copies the genes it needs right now — this is *selective compilation*.
3. **mRNA** is single-stranded, short-lived, and exported from the nucleus. It's the
   compiled bytecode that ribosomes can run.
4. **Translation** at the ribosome reads the mRNA sequence in triplets (codons) and
   assembles a chain of amino acids — the protein.
5. **Protein** does the work: enzyme, structural scaffold, signal receiver, motor, etc.

**What each genomics assay measures:**

| Assay | What it reads | What it misses |
|-------|--------------|----------------|
| WGS/WES | The source code (DNA sequence) | Whether genes are on or off |
| RNA-seq | Which mRNAs are present and how many | Protein levels, post-translational changes |
| Proteomics | Protein abundance | mRNA that isn't translated; regulation |
| ATAC-seq | Which parts of DNA are accessible | Sequence changes |
| ChIP-seq | Where a specific protein binds DNA | All other proteins |

**The critical subtlety:** mRNA abundance and protein abundance are only ~40–60%
correlated in mammalian cells. RNA-seq does not measure protein levels. This is why
RNA-seq findings always require validation.

In [ ]:
# Implementation: the genetic code as a lookup table
GENETIC_CODE = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

def translate(mrna: str) -> str:
    start = mrna.find('AUG')
    if start == -1: return ''
    protein = []
    for i in range(start, len(mrna) - 2, 3):
        aa = GENETIC_CODE.get(mrna[i:i+3], '?')
        if aa == '*': break
        protein.append(aa)
    return ''.join(protein)

mrna = 'AUGCUUAAAGGGUCAUGA'
print(f"mRNA:    {mrna}")
print(f"Protein: {translate(mrna)}")
print(f"\nThe genetic code maps {len(GENETIC_CODE)} codons → 20 amino acids + stop")

---
## 3. The Genome as a Database

The **genome** is the complete DNA sequence of an organism. Think of it as a very large,
hierarchically indexed, non-relational database:

```
Genome (3.2 billion base pairs)
├── Chromosome 1 (248 million bp)
│   ├── Gene: TP53   at position 7,668,421–7,687,490
│   │   ├── Promoter: −500 to −1 (transcription factor binding sites)
│   │   ├── Exon 1: 7,668,421–7,668,700 (coding sequence)
│   │   ├── Intron 1: 7,668,701–7,670,208 (spliced out)
│   │   └── Exon 2–11 ...
│   └── ...
└── Chromosome 2–22, X, Y, Mitochondria
```

**Scale comparison:**
- Human genome: ~3.2 billion bp (haploid)
- If printed as text at 12pt font: would fill ~1,200 volumes the size of a bible
- Only ~1.5% codes for protein — the rest is regulatory, repetitive, or 'junk'
  (though "junk" is a misnomer: ENCODE showed ~80% is transcribed)

**Genome coordinate system:**
Every position in the human genome has an address: `chr7:117,548,628`. This is like
a memory address — chromosome name + position. Variant callers report variants in
this format: `chr7:117,548,628 G>A` means the G at that position is changed to A.

In [ ]:
# Genome composition visualization
categories = [
    ('Protein-coding\nexons', 1.5, '#4e9af1'),
    ('Non-coding RNA\ngenes', 5.0, '#ff7043'),
    ('Regulatory\nelements', 8.0, '#66bb6a'),
    ('Introns', 26.5, '#ab47bc'),
    ('Repetitive\nelements', 50.0, '#ffa726'),
    ('Other /\nunclassified', 9.0, '#78909c'),
]

labels = [c[0] for c in categories]
sizes  = [c[1] for c in categories]
colors = [c[2] for c in categories]

fig, ax = plt.subplots(figsize=(9, 5))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct='%1.0f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(linewidth=1.5, edgecolor='white')
)
for t in texts: t.set_fontsize(9)
for at in autotexts: at.set_fontsize(8)

ax.set_title('Human genome composition: only 1.5% codes for protein\n'
             '(but ~80% is transcribed into RNA — ENCODE 2012)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. Mutation as Bit-Flip Errors

DNA is copied every time a cell divides. The copy is nearly perfect — the error rate is
~1.2 × 10⁻⁸ per base per generation (one error per 83 million base pairs copied).
With ~3.2 billion bases in the human genome, each cell division introduces ~40 new mutations.

**Types of mutations (analogies from computing):**

| Mutation | Computing analogy | Biological consequence |
|----------|------------------|------------------------|
| SNP (single base change) | Bit flip | May change amino acid; may be silent |
| Insertion/deletion (indel) | Off-by-one error | Often frameshifts the reading frame → garbage protein |
| Copy number variant | Memory segmentation error | May amplify an oncogene or delete a tumour suppressor |
| Translocation | Wrong function pointer | Two genes fused → novel (often oncogenic) protein |
| Chromosome non-disjunction | Array index out of bounds | Extra or missing chromosome (e.g. Down syndrome = trisomy 21) |

**Why the mutation rate matters:**
If the rate were 10× higher, too many essential functions would break. If 10× lower,
evolution would stall — no variation to select on. The current rate is near the
threshold where beneficial mutations can accumulate faster than harmful ones accumulate
(this is called the **mutation-selection balance**).

In [ ]:
# Demonstrate: frameshift is catastrophic; synonymous change is silent
original = 'AUGCCGAAUUGGAAAGCUUGA'

def show_translation_comparison(original, modified, label):
    print(f"  {label}")
    print(f"    Original:  {original} → {translate(original)}")
    print(f"    Modified:  {modified} → {translate(modified)}")
    print()

# 1. Synonymous SNP: last position of codon 2 (CCG→CCA, both = Pro)
syn_snp = original[:5] + 'A' + original[6:]
# 2. Missense SNP: CCG → CGG (Pro → Arg)
mis_snp = original[:3] + 'CGG' + original[6:]
# 3. Frameshift: insert one base at position 4
frameshift = original[:3] + 'A' + original[3:]

print("Mutation consequences:")
show_translation_comparison(original, syn_snp, 'Synonymous SNP (CCG→CCA, Pro→Pro)')
show_translation_comparison(original, mis_snp, 'Missense SNP (CCG→CGG, Pro→Arg)')
show_translation_comparison(original, frameshift, 'Frameshift insertion (+A after codon 1)')

---
## 5. Evolution as Optimization Without a Loss Function

Evolution is gradient-free optimization over a very high-dimensional fitness landscape,
but with no global objective. Each generation, variants are generated randomly (mutation
and recombination) and the environment selects which survive to reproduce. There is no
'target' — fitness is relative to the current environment, which changes.

**The optimization analogy:**

| Evolutionary concept | ML analogy |
|---------------------|------------|
| Genome | Model weights |
| Mutation | Gradient noise / random weight perturbation |
| Recombination (sex) | Crossover in genetic algorithms |
| Natural selection | Fitness evaluation (but no backprop — selection is forward-only) |
| Genetic drift | Stochastic gradient with small batch size |
| Speciation | Different optima in different loss landscapes |

**Why it's not gradient descent:**
- No gradient is computed — selection is blind to the landscape
- The landscape changes as the environment changes (moving target)
- 'Fitness' is not a fixed function — a trait that is advantageous in one environment
  (e.g. antibiotic resistance without antibiotics) may be costly in another
- Drift means some variants fix by chance, not by fitness

In [ ]:
# Fitness landscape analogy: allele frequency trajectory under different forces
rng = np.random.default_rng(123)

def simulate(N, p0, n_gen, s=0.0):
    p = p0
    traj = [p]
    for _ in range(n_gen):
        if s != 0:
            q = 1 - p
            p = p*(1+s)/(p*(1+s)+q)
        count = rng.binomial(2*N, p)
        p = count / (2*N)
        traj.append(p)
    return np.array(traj)

n_gen = 400
gen = np.arange(n_gen + 1)

fig, ax = plt.subplots(figsize=(10, 4))

# Multiple drift trajectories (small batch stochastic 'optimisation')
for _ in range(20):
    ax.plot(gen, simulate(50, 0.3, n_gen, s=0), color='lightgray', lw=0.8, alpha=0.7)

# Selection trajectory (gradient-like ascent)
sel_traj = np.array([np.mean([simulate(50, 0.3, n_gen, s=0.05) for _ in range(50)], axis=0)
                     for _ in range(1)])[0]
ax.plot(gen, sel_traj, color='tomato', lw=2.5, label='Mean trajectory with selection (s=0.05)')

ax.set_xlabel('Generation')
ax.set_ylabel('Allele frequency p')
ax.set_title('Evolution as optimization:\n'
             'Gray=drift alone (random walk); Red=with selection (biased walk toward p=1)')
ax.legend()
ax.axhline(0.3, color='gray', ls=':', lw=1, label='p₀')
plt.tight_layout()
plt.show()

---
## 6. Worked Example: Finding Drug Resistance Genes

**Biological question:** A bacterial pathogen evolved resistance to an antibiotic.
Which genes changed?

**Computational approach:**

1. Sequence the genome of resistant and sensitive strains (WGS)
2. Align reads to a reference genome (BWA-MEM)
3. Call variants (GATK HaplotypeCaller)
4. Filter variants: present in resistant, absent in sensitive
5. Annotate variants: which genes? what effect? (snpEff, VEP)
6. Prioritize: loss of function in known resistance genes first

**Why biology matters at each step:**
- Step 2: need to know genome structure to understand multi-mapping reads near repeats
- Step 4: need to know ploidy — bacteria are haploid, so a resistant allele at AF=0.5
  means a mixed culture, not a heterozygous individual
- Step 6: need to know which genes are known resistance determinants (e.g. rpoB for
  rifampicin; katG for isoniazid) to prioritize candidates

In [ ]:
# Simulated variant filtering pipeline
rng_ex = np.random.default_rng(42)

n_variants = 500
# Each variant: AF in sensitive, AF in resistant, gene, consequence
af_sensitive = rng_ex.beta(0.5, 10, n_variants)  # mostly rare in sensitive
af_resistant = rng_ex.beta(0.5, 10, n_variants)  # mostly rare background

# Plant a causal mutation: variant 42 is fixed in resistant, absent in sensitive
af_sensitive[42] = 0.0
af_resistant[42] = 1.0
consequence = ['synonymous'] * n_variants
consequence[42] = 'missense'
gene_labels = [f'gene_{i}' for i in range(n_variants)]
gene_labels[42] = 'rpoB'

# Filter: variants with high AF in resistant but not in sensitive
delta_af = af_resistant - af_sensitive
candidate_mask = delta_af > 0.7

print(f"Total variants: {n_variants}")
print(f"After filtering (ΔAF > 0.7): {candidate_mask.sum()} candidates")
for i in np.where(candidate_mask)[0]:
    print(f"  → {gene_labels[i]}: AF_sensitive={af_sensitive[i]:.3f}, "
          f"AF_resistant={af_resistant[i]:.3f}, consequence={consequence[i]}")

---
## Summary: Five Things to Remember

1. **The cell** is a factory running the same software (DNA) on specialized hardware (organelles)
2. **DNA → RNA → Protein** is the data pipeline. RNA-seq reads mRNA — not protein.
3. **The genome** is a 3.2-billion-character database; only 1.5% codes for proteins.
4. **Mutations** are copy errors. Frameshifts corrupt the reading frame catastrophically;
   synonymous changes are often neutral.
5. **Evolution** is forward-only, gradient-free optimization over a changing landscape.
   Drift and selection both change allele frequencies; drift dominates in small populations.

---
## Where to Go Next

If this notebook made sense, you are ready to:
- Read Module 08 (Bioinformatics: Sequence Analysis) — sequence alignment, BLAST
- Read Module 09 (Genomics and NGS) — the actual bioinformatics pipeline in detail
- Pick up *Bioinformatics Algorithms* (Compeau & Pevzner) — the standard CS-friendly intro

*This notebook is portfolio artifact MP02 from the 12-month Computational Biology
Research Notebook Program.*